## Figure 8 — connectedness of 2-hyperedges in real-world structures
---

Two real hypergraphs shipped in `HO-Chaotic-Oscillators/Real_structures/`: **Zachary Karate Club** and **cat-brain connectome**. Cliques in the original datasets are promoted to 2-hyperedges ⇒ $\mathcal I^{(1,2)}=1$.

- (b) Karate original vs (c) Karate **modified** (one extra 2-hyperedge added → triangle layer becomes connected → bounded region even at $\sigma_1\to0$).
- (e) Cat-brain original (connected) vs (f) Cat-brain **modified** (a node isolated at the 2-hyperedge level → disconnected → stability lost at $\sigma_1\to0$).

We first show the **cheap, decisive diagnostic** — the algebraic connectivity $\lambda_2(L^{(2)})$ of the triangle layer (0 ⇔ disconnected) — which is what actually drives the panels and needs no dynamics. Then, optionally, the full Rössler sweeps that reproduce the black regions themselves.


In [ ]:
import sys, os, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import seaborn as sns

sys.path.insert(0, os.getcwd())
import util
sys.path.insert(0, util.OVERLAP_DIR)
from hyperedge_overlap_utils import (intra_order_hyperedge_overlap as intra_overlap,
                                      inter_order_hyperedge_overlap as inter_overlap)
out_dir = Path(util.REPO_ROOT) / 'outputs' / 'fig8'
out_dir.mkdir(parents=True, exist_ok=True)
rng_seed = 1

In [7]:
def structure_at_overlap(N, k1, k2, target_T2, target_I12, seed=0):
    '''Approximate generator spanning (T2, I12) in [0,1]^2 for M=2.

    Strategy:
      * Triangles: interpolate between clustered blocks (T2~1) and random
        regular (T2~0) by assigning a fraction ~target_T2 of nodes to blocks.
      * Pairs: a fraction ~target_I12 of the required triangle-faces are added
        as links (raising inter-order overlap); the remaining links are random
        to keep k1 regular-ish.
    Returns pairs, triangles.
    '''
    rng = np.random.default_rng(seed)
    # ---- triangles: blend blocked and random ----
    # blocks of size 4 give k2 = C(3,2) = 3
    Bsz = util.block_size_for_degrees({2: k2})   # k2=3 -> B=4
    n_block_nodes = int(round(target_T2 * N))
    n_block_nodes -= n_block_nodes % Bsz         # multiple of block size
    tri_list = []
    if n_block_nodes >= Bsz:
        blk, _ = util.clustered_blocks(n_block_nodes, Bsz, orders=[2], seed=seed)
        tri_list.append(blk[2])
    # remaining nodes get random-regular triangles
    rem = np.arange(n_block_nodes, N)
    if len(rem) >= 3:
        rt, _ = util.random_regular_mbody(len(rem), k2, 2, seed=seed+1)
        tri_list.append(rt + n_block_nodes)
    triangles = np.vstack([t for t in tri_list if len(t)]) if tri_list else \
                util.random_regular_mbody(N, k2, 2, seed=seed)[0]
    # ---- pairs: mix triangle-faces (inter overlap) with random links ----
    faces = set()
    for a,b,c in triangles:
        faces.add(tuple(sorted((a,b)))); faces.add(tuple(sorted((a,c)))); faces.add(tuple(sorted((b,c))))
    faces = list(faces); rng.shuffle(faces)
    n_face_links = int(round(target_I12 * len(faces)))
    chosen = set(faces[:n_face_links])
    # top up with random links to approx k1-regular
    deg = np.zeros(N,int)
    pairs = set(chosen)
    for a,b in chosen: deg[a]+=1; deg[b]+=1
    target_links = k1*N//2
    tries=0
    while len(pairs) < target_links and tries < 200000:
        tries+=1
        i,j = rng.integers(0,N,2)
        if i==j: continue
        e=tuple(sorted((int(i),int(j))))
        if e in pairs: continue
        pairs.add(e); deg[i]+=1; deg[j]+=1
    return np.array(sorted(pairs),dtype=np.int64), triangles

# quick single-point sanity
p,t = structure_at_overlap(100,6,3,0.5,0.5,seed=0)
print("sanity:", intra_overlap(t,2)[0], inter_overlap(p,1,t,2))

sanity: 0.5 0.5111111111111111


In [5]:
# --- cheap diagnostic: triangle-layer connectivity for all four structures ---
names = ["Karate_club_original","Karate_club_modified",
         "Cat_brain_original","Cat_brain_modified"]
print(f"{'structure':28s} {'N':>4s} {'lambda2(L2)':>12s} {'#components':>12s}")
data = {}
for nm in names:
    d = util.load_real_structure(nm); data[nm]=d
    L2,_ = util._laplacian_fixed_N(d["triplets"],2,d["N"])
    w = np.sort(np.linalg.eigvalsh((L2+L2.T)/2))
    ncomp = int(np.sum(w < 1e-8))
    print(f"{nm:28s} {d['N']:>4d} {w[1]:>12.4f} {ncomp:>12d}")

structure                       N  lambda2(L2)  #components
Karate_club_original           34      -0.0000            3
Karate_club_modified           34       0.2741            1
Cat_brain_original             65       6.0496            1
Cat_brain_modified             65       0.0000            2


In [8]:
PRESET7 = "medium"   # "coarse" | "paper"
CFG7 = {"coarse": dict(n_steps=300, grid=13, n_resampling=100),
        "medium": dict(n_steps=400, grid=15, n_resampling=100),
        "paper":  dict(n_steps=1500, grid=25, n_resampling=200)}[PRESET7]
N=100; k1=6; k2=3
s1_7 = np.logspace(-6,0,CFG7["grid"]); s2_7 = np.logspace(-6,0,CFG7["grid"])

# (a) T2=0, I12=1  : random-ish triangles (connected), pairs = triangle faces
pa, ta = structure_at_overlap(N,k1,k2, target_T2=0.0, target_I12=1.0, seed=0)
# (b) T2=1, I12=1  : blocked triangles (disconnected), pairs include faces
pb, tb = structure_at_overlap(N,k1,k2, target_T2=1.0, target_I12=1.0, seed=0)
for tag,p,t in [("(a){0,1}",pa,ta),("(b){1,1}",pb,tb)]:
    print(tag,"T2=%.2f I12=%.2f"%(intra_overlap(t,2)[0], inter_overlap(p,1,t,2)))
    # connectedness of triangle layer:
    L2,_=util._laplacian_fixed_N(t,2,N); w=np.sort(np.linalg.eigvalsh((L2+L2.T)/2))
    print("     triangle-layer lambda2=%.3f (0 => disconnected)"%w[1])

(a){0,1} T2=0.03 I12=1.00
     triangle-layer lambda2=1.076 (0 => disconnected)
(b){1,1} T2=1.00 I12=1.00
     triangle-layer lambda2=-0.000 (0 => disconnected)


In [ ]:
PRESET8 = "medium"
CFG8 = {"coarse": dict(n_steps=300, grid=13, n_resampling=100),
        "medium": dict(n_steps=600, grid=15, n_resampling=150),
        "paper":  dict(n_steps=1500, grid=25, n_resampling=200)}[PRESET8]
s1_8 = np.logspace(-6,0,CFG8["grid"]);
s2_8 = np.logspace(-6,0,CFG8["grid"])

panels = {"(b) Karate original":"Karate_club_original",
            "(c) Karate modified":"Karate_club_modified",
            "(e) Cat original":"Cat_brain_original",
            "(f) Cat modified":"Cat_brain_modified"}

for i,(ttl,nm) in enumerate(panels.items()):
    t0=time.time()
    d=data[nm]
    err,_ = util.rossler_error_grid(d["N"], d["list_neighbors"], d["triplets"],
        s1_8, s2_8, h=1e-3, n_steps=CFG8["n_steps"],
        n_resampling=CFG8["n_resampling"], coupling="type3", seed=0, verbose=False)
    print(ttl,f"done in {time.time()-t0:.1f}s")
    np.save(out_dir / f'fig8-err{i+1}.npy', err)

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(10,9), constrained_layout=True)
for i,(ax,ttl) in enumerate(zip(axes.flat, panels.keys())):
    err = np.load(out_dir / f'fig8-err{i+1}.npy')
    region=(err<1.0).astype(float)
    ax.pcolormesh(s1_8,s2_8,region,shading="auto",cmap=plt.colormaps['mako_r'])
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$\sigma_1$")
    ax.set_ylabel(r"$\sigma_2$")
    ax.set_title(ttl)
    ax.set_box_aspect(1)
    ax.set_xlim([1e-6, 1])
    ax.set_ylim([1e-6, 1])
    ax.set_xticks([1, 1e-2, 1e-4, 1e-6])
    ax.set_yticks([1, 1e-2, 1e-4, 1e-6])
fig.suptitle("Fig 8 (b,c,e,f) -- stability regions in real structures")
plt.show()